# Aggregation, comparisons and sensitivity

Curated excerpts from the original `Code/SpaceMeanSpeed.ipynb` and `Code/Travel Time Cal - V2-For use.ipynb`. Source cell numbers are recorded in each code cell. Research data and outputs are not included.

The first calculation implements space-mean speed as **trip count × segment length / total travel time**, rather than the arithmetic mean of per-trip speeds. It retains the original 1,606 m segment and hourly bins. Comparison and KS sections use prepared GTFS/Bluetooth/Google Traffic tables; the final section retains the original buffer-sensitivity correlation calculation. These are independent analysis sections with their own input workbooks, not a sequential raw-data reproduction.

Append-mode workbook export cells were omitted. Input paths and the figure output path were replaced by configurable locations. No statistical formulas were changed. The original KS interpretation text is preserved; failure to reject does not establish distributional equivalence.


In [ ]:
# Portfolio adapter: configure authorized inputs outside the repository.
from pathlib import Path
import os
if not os.environ.get("GTFS_DATA_ROOT"):
    raise RuntimeError("Set GTFS_DATA_ROOT to a directory containing your authorized input files.")
DATA_ROOT = Path(os.environ["GTFS_DATA_ROOT"]).expanduser()
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# Original research code: Code/SpaceMeanSpeed.ipynb, cell 0 (zero-based).
import pandas as pd
import geopandas as gpd
from shapely import geometry
from shapely.geometry import Point, LineString, shape, MultiLineString
import matplotlib.pyplot as plt
import contextily as ctx
import datetime
import matplotlib.dates as mdates
from matplotlib.patches import Circle, FancyArrow
import matplotlib.patches as mpatches
import numpy as np
import math
import statistics
from geopy.distance import geodesic
import os
import json
from matplotlib.dates import date2num, HourLocator, DateFormatter
from datetime import datetime
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy import stats
import random

import warnings
warnings.filterwarnings('ignore')

## Space-mean speed from trip travel times

Provide `trip_travel_times.xlsx`, sheet `Seg 6 (5)`, with `timestamps` and `time_costs` (seconds). The original segment length is 1,606 m; do not substitute the other notebook’s 1,673 m output without adapting the configuration.

In [ ]:
# Original research code: Code/SpaceMeanSpeed.ipynb, cell 2 (zero-based).
spd_data = pd.read_excel(str(DATA_ROOT / "trip_travel_times.xlsx"), sheet_name="Seg 6 (5)")
spd_data['timestamps'] = pd.to_datetime(spd_data['timestamps'])
spd_data.set_index('timestamps', inplace=True)
length = 1606
result = spd_data.resample('H').apply(lambda x: (len(x) * length) / x['time_costs'].sum())


## Compare traffic sources

Provide `comparison_speed.xlsx`, sheet `10.10`, with `Time`, `GTFS Realtime`, `Bluetooth`, and `Google Traffic` (speeds in m/s). The original plot title is preserved as a historical label and should be updated to match actual inputs.

In [ ]:
# Original research code: Code/SpaceMeanSpeed.ipynb, cell 17 (zero-based).
comparison_seg = pd.read_excel(str(DATA_ROOT / "comparison_speed.xlsx"), sheet_name='10.10')

In [ ]:
# Original research code: Code/SpaceMeanSpeed.ipynb, cell 19 (zero-based).
plt.figure(figsize=(15, 7))

x = comparison_seg['Time'].astype(str)

# 绘制三条折线，分别代表GTFS Realtime、Bluetooth、Google Traffic
plt.plot(x, comparison_seg['GTFS Realtime'], label='GTFS Realtime', marker='o', linestyle='-', color='blue', alpha=0.5)
plt.plot(x, comparison_seg['Bluetooth'], label='Bluetooth', marker='^', linestyle=':', color='grey')
plt.plot(x, comparison_seg['Google Traffic'], label='Google Traffic', marker='*', linestyle='--', color='black')

plt.legend(fontsize=14)
plt.xlabel('Time', fontsize=14)
plt.ylabel('Speed (m/s)', fontsize=14)
plt.xticks(rotation=45, fontsize=10)
plt.ylim(0)

plt.title('2023-10-26',  fontsize=14)
plt.grid(True, linestyle='-', alpha=0.3)

plt.savefig(
    str(OUTPUT_DIR / "speed_comparison.png"),
    dpi=300, 
    bbox_inches='tight'
)
plt.show()
plt.close()


## Distribution comparison

Provide `comparison_speed_for_ks.xlsx`, sheet `10.12`, with the three speed columns. This retains the source mean-shifting/normalization and GTFS-versus-Google Traffic comparison.

In [ ]:
# Original research code: Code/SpaceMeanSpeed.ipynb, cell 21 (zero-based).
data_dist = pd.read_excel(
    str(DATA_ROOT / "comparison_speed_for_ks.xlsx"), 
    sheet_name='10.12'
)

data_dist['GTFS Realtime_4_Bluetooth'] = data_dist['GTFS Realtime'] - data_dist['GTFS Realtime'].sum()/data_dist['GTFS Realtime'].shape[0] + data_dist['Bluetooth'].sum()/data_dist['Bluetooth'].shape[0]
data_dist['GTFS Realtime_4_Google_Traffic'] = data_dist['GTFS Realtime'] - data_dist['GTFS Realtime'].sum()/data_dist['GTFS Realtime'].shape[0] + data_dist['Google Traffic'].sum()/data_dist['Google Traffic'].shape[0]

# Generate two sample data arrays
GTFS_Realtime_4_Bluetooth = data_dist['GTFS Realtime_4_Bluetooth']/data_dist['GTFS Realtime_4_Bluetooth'].sum()
GTFS_Realtime_4_GoogleTraffic = data_dist['GTFS Realtime_4_Google_Traffic']/data_dist['GTFS Realtime_4_Google_Traffic'].sum()
Bluetooth = data_dist['Bluetooth']/data_dist['Bluetooth'].sum()
Google_Traffic = data_dist['Google Traffic']/data_dist['Google Traffic'].sum()

In [ ]:
# Original research code: Code/SpaceMeanSpeed.ipynb, cell 22 (zero-based).
import numpy as np
from scipy.stats import ks_2samp

# Perform a two-sample KS test
statistic, p_value = ks_2samp(GTFS_Realtime_4_GoogleTraffic, Google_Traffic)

print(f"KS Statistic: {statistic}")
print(f"P-value: {p_value}")

# Interpret the results
alpha = 0.05
if p_value < alpha:
    print("Reject the null hypothesis: The two samples have different distributions.")
else:
    print("Fail to reject the null hypothesis: The two samples have similar distributions.")

## Stop-buffer sensitivity

Provide `buffer_sensitivity.xlsx`, sheet `Segment 1 - buffer size`, with `GTFS Realtime - 10m`, `GTFS Realtime - 20m`, `GTFS Realtime - 30m`, `Bluetooth`, and `Google Traffic`.

In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 169 (zero-based).
data_linreg = pd.read_excel(str(DATA_ROOT / "buffer_sensitivity.xlsx"), sheet_name='Segment 1 - buffer size')

In [ ]:
# Original research code: Code/Travel Time Cal - V2-For use.ipynb, cell 171 (zero-based).
x1 = np.array(data_linreg['GTFS Realtime - 10m'])
x2 = np.array(data_linreg['GTFS Realtime - 20m'])
x3 = np.array(data_linreg['GTFS Realtime - 30m'])
y1 = np.array(data_linreg['Bluetooth'])
y2 = np.array(data_linreg['Google Traffic'])

slope, intercept = np.polyfit(x3, y1, 1)
correlation_coefficient, _ = stats.pearsonr(x3, y1)

print("Slope:", slope)
print("Intercept:", intercept)
print("R:", correlation_coefficient)



# plt.figure(figsize=(15, 7))

# equation_label = f'Linear Equation: y = {slope:.2f}x + {intercept:.2f}'
# correlation_label = f'Correlation (R): {correlation_coefficient:.3f}'

# plt.text(13, 6.5, equation_label, fontsize=14, color='black')
# plt.text(13, 6, correlation_label, fontsize=14, color='black')

# plt.scatter(x, y1, label='Data Points')
# plt.plot(x, slope * x + intercept, color='red', label='Linear Regression Line')
# plt.xticks(fontsize=12)
# plt.yticks(fontsize=12)
# plt.legend(fontsize=14)
# plt.xlabel('GTFS Realtime (m/s)', fontsize=14)
# plt.ylabel('Bluetooth (m/s)', fontsize=14)


# plt.show()